In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(67)

def load_and_clean(filename):
    return np.loadtxt(filename, skiprows=2)

data_x = load_and_clean('x24x24.txt')
data_y = load_and_clean('y24x24.txt')
data_z = load_and_clean('z24x24.txt')
full_data = np.vstack([data_x, data_y, data_z])

X = full_data[:, :576]
y = full_data[:, 578]

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=67,
    stratify=y           # photos of a person will be in both sets
)

In [2]:
from data_augumentation import augment_df
X_new, Y_new = augment_df(X_train, y_train, min_target=110, max_target=155)
unique, c = np.unique(Y_new, return_counts=True)
c

array([122, 150, 110, 110, 110, 110, 155, 110, 147, 110, 110, 147, 126,
       155, 110, 110, 110, 110, 113, 110, 110, 122, 110, 122, 155, 125,
       118, 146, 155, 110, 110, 110, 110, 149, 114, 110, 110, 110, 110,
       110, 146, 136, 155, 155, 110, 137, 138, 110])

In [5]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.85, random_state=67)
X_train_pca = pca.fit_transform(X_new)
X_val_pca = pca.transform(X_val)

print(pca.explained_variance_ratio_.sum())
print(pca.n_components_)


0.8502582282677102
107


In [6]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

for depth in [3, 5, 7, 10, 12]:
    weak = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_leaf=3,
        min_samples_split=5,
        random_state=67
    )

    clf = AdaBoostClassifier(
        estimator=weak,
        n_estimators=200,
        learning_rate=0.05,
        random_state=67
    )

    clf.fit(X_train_pca, Y_new)
    print(depth, clf.score(X_train_pca, Y_new), clf.score(X_val_pca, y_val))

3 0.3068373189626137 0.2757863935625457
5 0.6081172111822163 0.43452816386247256
7 0.9590771303469181 0.5281638624725676
10 1.0 0.5991221653255303
12 1.0 0.6364301389904902
